In [1]:
import os
import json
import spacy
import nltk
from spacy_wordnet.wordnet_annotator import WordnetAnnotator
from nltk.corpus import wordnet

In [2]:
nlp =  spacy.load("en_core_web_sm")
nlp.add_pipe("spacy_wordnet", after='tagger')
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
file_path = os.path.join("..","data","stories.json")
print(file_path)

../data/stories.json


In [4]:
stories =[]

In [5]:
with open(file_path, "r", encoding="utf-8") as f:
    stories = json.load(f)

print(stories)    

{'title': 'rabbit, weasel, cat', 'actors': ['rabbit', 'weasel', 'cat'], 'storyMoral': 'The strong can settle questions to their own advantage', 'englishVersion': 'When the rabbit left his home, a weasel moved in and refused to move out again. A great dispute arose over this.  When the cat heard this, she offered to pacify the dispute. She asked the opponents to come closer to her, because she could not hear well. When this happened, she grabbed one of the disputants with each paw to devour them. Thus she pacified the quarrel.', 'transliteratedVersion': ['zazakaH nakulaH mArjArI ca', 'ekadine gRhAt prasthitaH ekaH zazakaH svAduM tRNaM khAditum agacchat ।', 'parantu gRhasya pidhAnaM kartuM saH vismRtavAn ।', 'yadA zazakaH gRhe na AsIt tadA ekaH nakulaH tatra Agatya zazakasya gRhaM praviSTavAn ।', 'tasmai zazakasya gRham arocata ataH tatraiva nivasituM nizcayaM kRtavAn ।', 'zazakaH Agatya nakulaM dRSTvA ca avadat –', '" mama gRhAt nirgacchatu zIghram " – iti ।', 'parantu nakulaH gantuM na

In [6]:
tokenized_version = stories["tokenized_english_version"]
print(tokenized_version)

[{'text': 'When', 'lemma': 'when', 'pos': 'SCONJ'}, {'text': 'the', 'lemma': 'the', 'pos': 'DET'}, {'text': 'rabbit', 'lemma': 'rabbit', 'pos': 'NOUN'}, {'text': 'left', 'lemma': 'leave', 'pos': 'VERB'}, {'text': 'his', 'lemma': 'his', 'pos': 'PRON'}, {'text': 'home', 'lemma': 'home', 'pos': 'NOUN'}, {'text': ',', 'lemma': ',', 'pos': 'PUNCT'}, {'text': 'a', 'lemma': 'a', 'pos': 'DET'}, {'text': 'weasel', 'lemma': 'weasel', 'pos': 'NOUN'}, {'text': 'moved', 'lemma': 'move', 'pos': 'VERB'}, {'text': 'in', 'lemma': 'in', 'pos': 'ADV'}, {'text': 'and', 'lemma': 'and', 'pos': 'CCONJ'}, {'text': 'refused', 'lemma': 'refuse', 'pos': 'VERB'}, {'text': 'to', 'lemma': 'to', 'pos': 'PART'}, {'text': 'move', 'lemma': 'move', 'pos': 'VERB'}, {'text': 'out', 'lemma': 'out', 'pos': 'ADP'}, {'text': 'again', 'lemma': 'again', 'pos': 'ADV'}, {'text': '.', 'lemma': '.', 'pos': 'PUNCT'}, {'text': 'A', 'lemma': 'a', 'pos': 'DET'}, {'text': 'great', 'lemma': 'great', 'pos': 'ADJ'}, {'text': 'dispute', 'le

In [12]:
tokens = tokenized_version

In [13]:
def get_wordnet_pos(pos):
    if pos == "NOUN":
        return wordnet.NOUN
    elif pos == "VERB":
        return wordnet.VERB
    elif pos == "ADJ":
        return wordnet.ADJ
    elif pos == "ADV":
        return wordnet.ADV
    else:
        return None

In [14]:
def get_synonyms(word, wn_pos):
    synonyms = set()

    for syn in wordnet.synsets(word, pos=wn_pos):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name())

    return list(synonyms)

In [15]:
def get_antonyms(word, wn_pos):
    antonyms = set()

    for syn in wordnet.synsets(word, pos=wn_pos):
        for lemma in syn.lemmas():
            for ant in lemma.antonyms():
                antonyms.add(ant.name())

    return list(antonyms)

In [18]:
updated_tokens = []

for token in tokenized_version:

    lemma = token["lemma"]
    pos = token["pos"]

    wn_pos = get_wordnet_pos(pos)

    # skip unsupported tokens
    if wn_pos is None or not lemma.isalpha():

        token["synonyms"] = []
        token["antonyms"] = []

        updated_tokens.append(token)
        continue

    # merge into EXISTING token
    token["synonyms"] = get_synonyms(lemma, wn_pos)
    token["antonyms"] = get_antonyms(lemma, wn_pos)

    updated_tokens.append(token)

In [19]:
for r in results[:10]:
    print(r)

{'word': 'rabbit', 'pos': 'NOUN', 'synonyms': ['rabbit', 'lapin', 'coney', 'cony', 'hare'], 'antonyms': []}
{'word': 'leave', 'pos': 'VERB', 'synonyms': ['impart', 'provide', 'go_away', 'pass_on', 'give', 'get_out', 'leave_alone', 'lead', 'bequeath', 'entrust', 'leave_behind', 'go_forth', 'allow_for', 'allow', 'pull_up_stakes', 'forget', 'result', 'go_out', 'will', 'leave', 'exit', 'depart'], 'antonyms': ['disinherit', 'enter', 'arrive']}
{'word': 'home', 'pos': 'NOUN', 'synonyms': ['dwelling', 'home_base', 'menage', 'domicile', 'rest_home', 'nursing_home', 'family', 'place', 'dwelling_house', 'household', 'base', 'house', 'home_plate', 'abode', 'home', 'habitation', 'plate'], 'antonyms': []}
{'word': 'weasel', 'pos': 'NOUN', 'synonyms': ['weasel'], 'antonyms': []}
{'word': 'move', 'pos': 'VERB', 'synonyms': ['motivate', 'displace', 'impress', 'strike', 'prompt', 'act', 'travel', 'run', 'locomote', 'make_a_motion', 'affect', 'propel', 'go', 'proceed', 'incite', 'actuate', 'move', 'be_a

In [20]:
stories["tokenized_english_version"] = updated_tokens

In [21]:
print(stories)

{'title': 'rabbit, weasel, cat', 'actors': ['rabbit', 'weasel', 'cat'], 'storyMoral': 'The strong can settle questions to their own advantage', 'englishVersion': 'When the rabbit left his home, a weasel moved in and refused to move out again. A great dispute arose over this.  When the cat heard this, she offered to pacify the dispute. She asked the opponents to come closer to her, because she could not hear well. When this happened, she grabbed one of the disputants with each paw to devour them. Thus she pacified the quarrel.', 'transliteratedVersion': ['zazakaH nakulaH mArjArI ca', 'ekadine gRhAt prasthitaH ekaH zazakaH svAduM tRNaM khAditum agacchat ।', 'parantu gRhasya pidhAnaM kartuM saH vismRtavAn ।', 'yadA zazakaH gRhe na AsIt tadA ekaH nakulaH tatra Agatya zazakasya gRhaM praviSTavAn ।', 'tasmai zazakasya gRham arocata ataH tatraiva nivasituM nizcayaM kRtavAn ।', 'zazakaH Agatya nakulaM dRSTvA ca avadat –', '" mama gRhAt nirgacchatu zIghram " – iti ।', 'parantu nakulaH gantuM na

In [22]:
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(stories, f, indent=4, ensure_ascii=False)